<center><h1>Annotations for the Dataset Generator</h1></center>
<center><h4>Going over all the code from everyone and splitting it up into a legible and usable base to generate the artifical markets for the Black-Sholes Training</h4></center>

### Imports

In [117]:
import numpy as np
import pandas as pd
from scipy.stats import t as t_dist
from arch import arch_model
import yfinance as yf
import time
import os
import json
import hashlib
from cir_model import CIRModel
from scipy.stats import truncnorm, norm
import pywt

### Miscellaneous Functions and Global Variables for Reproducibility

In [118]:
MASTER_SEED = 42  # Change this to generate different datasets

def set_all_seeds(seed):
    """Set all random seeds for reproducibility"""
    np.random.seed(seed)
    print(f"All seeds set to: {seed}")

def compute_dataframe_hash(df, sample_size=10000):
    """
    Compute hash of dataframe to verify reproducibility.
    Uses a sample to avoid memory issues with large datasets.
    """
    if len(df) > sample_size:
        df_sample = df.sample(n=sample_size, random_state=42).sort_index()
    else:
        df_sample = df
    
    return hashlib.md5(pd.util.hash_pandas_object(df_sample).values).hexdigest()

def save_metadata(metadata, filename='dataset_metadata.json'):
    """Save generation metadata for verification"""
    with open(filename, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"\nMetadata saved to {filename}")

#### Setting the seed in order to reproduce the same numbers

In [119]:
# Set seed before any random operations
set_all_seeds(MASTER_SEED)

start_date = "2019-01-01"
end_date = "2025-01-01"

All seeds set to: 42


### Downloading FTSE 100 Market Data and Fitting Garch model
Using a Garch(1,1)-t model, which is defined with returns as
$$r_t=\mu + \varepsilon_t$$
with the residuals
$$\varepsilon_t=\sigma_tz_t$$
and the resurcive variance equation
$$\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$
which put into infinite form is
$$\sigma_t^2=\frac{\omega}{1-\beta}+\sum_{k=1}^{\infty}\alpha\beta^{k-1}\varepsilon_{t-k}^2$$
Given $\sigma^2_t = $ conditional variance, and
$$z_t\sim \text{i.i.d. } t_v(0,1)$$
and the constraints
$$\omega > 0\\ \alpha \geq 0 \\ \beta \geq 0\\ \alpha + \beta < 1$$
So in total for parameters, we have
$$\theta = (\mu, \omega, \alpha, \beta, v)$$

In [120]:
data = yf.download("^FTSE", start=start_date, end=end_date)

if isinstance(data.columns, pd.MultiIndex):
    close_prices = data['Close']['^FTSE']
else:
    close_prices = data['Close']

S0 = float(close_prices.iloc[-1])

print(f"Downloaded {len(close_prices):,} trading days")
print(f"S_0 (initial stock price): £{S0:,.2f}")

# Fit GARCH(1,1)-t
returns_pct = 100 * np.log(close_prices / close_prices.shift(1)).dropna()

print("\nFitting GARCH(1,1)-t model...")
model = arch_model(returns_pct, vol='Garch', p=1, q=1, dist='t')
result = model.fit(disp='off')

mu = result.params['mu']
omega = result.params['omega']
alpha = result.params['alpha[1]']
beta = result.params['beta[1]']
nu = result.params['nu']

print(f"Parameters: μ={mu:.4f}%, ω={omega:.6f}, α={alpha:.4f}, β={beta:.4f}, ν={nu:.2f}")

[*********************100%***********************]  1 of 1 completed

Downloaded 1,514 trading days
S_0 (initial stock price): £8,173.00

Fitting GARCH(1,1)-t model...
Parameters: μ=0.0484%, ω=0.046704, α=0.1335, β=0.8205, ν=4.60


### Parameterizing a Cox–Ingersoll–Ross Model 
The CIR model is defined by the SDE
$$dr_t = \kappa(\theta - r_t)dt + \sigma\sqrt{r_t}dW_t$$
where $\kappa(\theta - r_t)$ serves as the drift term with a long-run mean of $\theta$ and a mean reversion speed of $\kappa$. $W_t$ is a standard Wiener process.

The volatility term $\sigma\sqrt{r_t}$ creates level dependent volatility, which dampens volatility when $r_t$ is close to 0. If the Feller condition is met, defined by
$$2\kappa \theta \geq \sigma ^2$$
then negative interest rates are impossible.

In the code we parameterize our CIR model on past daily SONIA data provided by the U.S. Federal Reserve, previously downloaded

In [121]:
# Getting a dataframe of historical interest rates
historical_IR_sonia = pd.read_csv("parameterize_data\IUDSOIA.csv")
historical_IR_sonia["observation_date"] = pd.to_datetime(historical_IR_sonia["observation_date"])
historical_ir_wanted = historical_IR_sonia[(historical_IR_sonia["observation_date"] >= start_date) & (historical_IR_sonia["observation_date"] <= end_date)].dropna()

print(historical_ir_wanted.head())
# Using the CIR Model Class in the repo
cir = CIRModel(historical_ir_wanted)
cir.calibrate(method='mle')

     observation_date  interest_rate
5739       2019-01-02         0.7044
5740       2019-01-03         0.7048
5741       2019-01-04         0.7046
5742       2019-01-07         0.7052
5743       2019-01-08         0.7052
Loaded 1515 data points
Date range: 2019-01-02 00:00:00 to 2024-12-31 00:00:00
Mean rate: 200.2999%
MLE optimization failed, falling back to method of moments

Calibrated parameters (Method of Moments):
  κ (kappa): 0.0557 - mean reversion speed
  θ (theta): 200.2999% - long-term mean
  σ (sigma): 0.4673 - volatility
  Feller condition (2κθ > σ²): 0.223076 > 0.218387 = True


{'kappa': 0.05568541895992372,
 'theta': 2.002999471947195,
 'sigma': 0.4673194433258763}

### Saving the model parameters in order to load them for later without using having to reparameterize the models

In [122]:
params = {
    "End Price" : S0,
    "Mean IR" : cir.historical_data['interest_rate'].mean(),
    "IR std" : np.std(cir.historical_data['interest_rate']),
    "GARCH" : {
        "mu": mu,
        "omega": omega,
        "alpha": alpha,
        "beta": beta,
        "nu": nu
    },
    "CIR": {
        "kappa": cir.kappa,
        "theta": cir.theta,
        "sigma": cir.sigma
    }
}

with open("parameterize_data/model_params.json", "w") as f:
    json.dump(params, f, indent=4)
    
del mu, omega, alpha, beta, nu, cir, data

print("model_params.json written successfully.")

model_params.json written successfully.


### Reloading the model simulation parameters, so code can be executed beyond this line

In [123]:
with open("parameterize_data/model_params.json", "r") as f:
    params = json.load(f)

# Historical Parameters
S0 = params["End Price"]
ir_mean = params["Mean IR"]
ir_std = params["IR std"]


# GARCH parameters
mu    = params["GARCH"]["mu"]
omega = params["GARCH"]["omega"]
alpha = params["GARCH"]["alpha"]
beta  = params["GARCH"]["beta"]
nu     = params["GARCH"]["nu"]

# CIR parameters
kappa = params["CIR"]["kappa"]
theta = params["CIR"]["theta"]
sigma = params["CIR"]["sigma"]

print("Option Price")
print(f"  S0 = {S0}")

print("\nInterest Rates")
print(f"  Mean Interest Rate = {ir_mean}")
print(f"  Interest Rate Standard Deviation = {ir_std}")

print("\nGARCH Parameters:")
print(f"  mu    (μ) = {mu}")
print(f"  omega (ω) = {omega}")
print(f"  alpha (α) = {alpha}")
print(f"  beta  (β) = {beta}")
print(f"  nu    (ν) = {nu}")

print("\nCIR Parameters:")
print(f"  kappa (κ) = {kappa}")
print(f"  theta (θ) = {theta}")
print(f"  sigma (σ) = {sigma}")

Option Price
  S0 = 8173.0

Interest Rates
  Mean Interest Rate = 2.002999471947195
  Interest Rate Standard Deviation = 2.1061137312665377

GARCH Parameters:
  mu    (μ) = 0.04842674175412159
  omega (ω) = 0.04670376471231804
  alpha (α) = 0.1334992945018068
  beta  (β) = 0.8204776734410293
  nu    (ν) = 4.6037768961503795

CIR Parameters:
  kappa (κ) = 0.05568541895992372
  theta (θ) = 2.002999471947195
  sigma (σ) = 0.4673194433258763


### Setting up the Simulation Parameters and Saved Metadata for reproducibility 

In [124]:
n_simulations = 10000
T_maturity = 5  # All simulations start at 5 years
K_percentages = np.array([0.60, 0.70, 0.80, 0.90, 1.00, 1.10, 1.20, 1.30, 1.40, 1.50])
days_per_year = 252
n_days = (T_maturity - 1) * days_per_year + 1 # Total days for each simulation
mu_adjusted = 0.04
min_starting_ir = 0.01
burn_in_period = 90

print(f"Total simulations: {n_simulations:,}")
print(f"Maturity (T): {T_maturity} years for ALL simulations")
print(f"Days per simulation: {n_days}")
print(f"K choices: {K_percentages * 100}%")

# Store metadata for reproducibility
metadata = {
    'master_seed': MASTER_SEED,
    'n_simulations': int(n_simulations),
    'T_maturity': int(T_maturity),
    'K_percentages': K_percentages.tolist(),
    'S0': float(S0),
    'days_per_year': int(days_per_year),
    'n_days': int(n_days),
    'mu_adjusted': float(mu_adjusted),
    'burn_in_period' : burn_in_period,
    'data_download': {
        'ticker': '^FTSE',
        'start_date': start_date,
        'end_date': end_date,
        'n_days': int(len(close_prices))
    },
    'GARCH_params': {
        "mu": mu,
        "omega": omega,
        "alpha": alpha,
        "beta": beta,
        "nu": nu
    },
    "CIR_params" : {
        "kappa": kappa,
        "theta": theta,
        "sigma": sigma
    },
    "IR_params" : {
        "IR Mean" : ir_mean,
        "IR Standard Deviation" : ir_std,
        "Minimum IR Start" : min_starting_ir
    },
    'python_version': os.sys.version,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'generation_timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

Total simulations: 10,000
Maturity (T): 5 years for ALL simulations
Days per simulation: 1009
K choices: [ 60.  70.  80.  90. 100. 110. 120. 130. 140. 150.]%


### Creating Strike Prices for the Whole Dataset

In [125]:
# CRITICAL: Set seed before random assignments
np.random.seed(MASTER_SEED)

# Each simulation gets ONE randomly assigned strike price
K_pct_assignments = np.random.choice(K_percentages, size=n_simulations)
K_assignments = S0 * K_pct_assignments

print(f"K distribution:")
for k_pct in K_percentages:
    count = np.sum(K_pct_assignments == k_pct)
    print(f"  K={k_pct*100:.0f}%: {count:,} ({count/n_simulations*100:.1f}%)")

K distribution:
  K=60%: 1,053 (10.5%)
  K=70%: 985 (9.8%)
  K=80%: 996 (10.0%)
  K=90%: 971 (9.7%)
  K=100%: 962 (9.6%)
  K=110%: 1,021 (10.2%)
  K=120%: 1,017 (10.2%)
  K=130%: 967 (9.7%)
  K=140%: 994 (9.9%)
  K=150%: 1,034 (10.3%)


### Simulating GARCH Price paths for the whole dataset
What this code is doing is creating two seperate multidimensional arrays through numpy to track volatility and price over time. We have two matrices $S_{ij}$ and $\sigma_{ij}$. Where $i$ is the number of scenarios we are running and $j$ is the number of days per scenario. Our initial volatility is given by
$$ \sigma^2_0 = \frac{\omega}{1-\alpha - \beta} $$
which is the expectation of variance. We set our initial positions for the simulations at 
$$\sigma_{i0} = \sqrt{\sigma^2}, \text{ as defined above}$$
$$S_{i0}=S0, \text{ the end of the stock data price time series we parameterized the GARCH on}$$
A Monte Carlo simulation is now performed using the GARCH, where
$$z_{i,t} \sim t_v$$
Which we normalize so
$$\text{Var}(z_{i,t})=1$$
Which we do by letting
$$z_{i,t}=\frac{\tilde{z}_{i,t}}{\sqrt{\frac{v}{v-2}}}$$
Since the normal t distribution variance is given by
$$\text{Var}(t_v) = \frac{v}{v-2}$$
The rest is just calculating the current variables using the equations described earlier for GARCH. Two things to note however are that
1. Volatility is capped at 25%
2. An additive log return process is used for numerical stability. Where instead of a multiplicative process
$$S_t = S_{t-1}e^{r_t}$$
we use
$$\log S_t = \log S_{t-1}+r_t$$
where
$$r_t = \mu + \varepsilon_t$$
and at the very end we take
$$S_t = e^{\log S_t}$$
What this does is uses a additive process instead of a multiplicative process the whole time which avoids floating point errors that come heavy multiplication

In [126]:
def simulate_garch(S0, n_days, n_scenarios, mu, omega, alpha, beta, nu, seed=None, burn_in=90):
    """
    High-performance GARCH(1,1)-t simulation
    using log-prices for numerical stability,
    90-day burn-in, and final rounding to 4 decimal places.
    """
    
    if seed is not None:
        np.random.seed(seed)
    
    # Adding a burn in period
    total_days = n_days + burn_in
    
    # Pre-draw standardized shocks for performance
    z = t_dist.rvs(df=nu, size=(n_scenarios, total_days))
    z /= np.sqrt(nu / (nu - 2)) # Standardizing 
    
    # Allocate arrays, multidimensional arrays
    var = np.empty((n_scenarios, total_days))
    log_S = np.empty((n_scenarios, total_days))
    
    # Initial conditions
    var0 = omega / (1 - alpha - beta)
    var[:, 0] = var0
    log_S[:, 0] = np.log(S0)
    
    
    # Time recursion
    for t in range(1, total_days):
        sigma_prev = np.sqrt(var[:, t-1])
        eps = sigma_prev * z[:, t]
        
        # Log-return update (additive)
        log_S[:, t] = log_S[:, t-1] + (mu + eps) / 100
        
        # Variance update
        var[:, t] = omega + alpha * eps**2 + beta * var[:, t-1]
        var[:, t] = np.minimum(var[:, t], 25.0) # Capping volatility 
    
    # Get final sigma
    sigma_final = np.sqrt(var[:,:])
    
    # Convert to prices
    S_final = np.exp(log_S)
    
    
    
    return S_final, sigma_final

### Simulating interest rates using CIR
Some notes about the methods:
1. The starting interest rate is sampled from a truncated normal distribution between (0.01, 25). 
$$X_{i,0} = \text{TruncNormal}(\mu_0, \sigma^2_0; 0.01, 25.0)$$
Which means it has a PDF of
$$f(x) = \frac{\phi\left(\frac{x-\mu_0}{\sigma_0}\right)}{\sigma\left[\Phi(b)-\Phi(a)\right]} \qquad \text{for $x \in [0.01, 10]$}$$
where in this case
$$a = \Phi\left(\frac{0.01-\mu_0}{\sigma_0}\right), \qquad b=\Phi\left(\frac{25-\mu_0}{\sigma_0}\right)$$
2. Using the Euler-Maruyama Method
$$dr_t = \kappa (\theta - r_t)dt+\sigma\sqrt{r_t}dW_t$$
we turn into
$$r_{t+1}=r_t+\kappa(\theta-r_t)\Delta t + \sigma\sqrt{r_t}\sqrt{\Delta t}Z_t$$
where
$$Z_t \sim N(0,1)$$

In [127]:
def simulate_CIR_paths(n_scenarios, n_steps, kappa, theta, sigma, mu0, sigma0_init, dt=1/252,burn_in_days=90, seed=None):
    if seed is not None:
        np.random.seed(seed)
    
    # Convert burn-in days to number of steps
    burn_in_steps = int(burn_in_days / (dt * 252)) if dt != 1/252 else burn_in_days
    
    print(f"Total burn in steps: {burn_in_steps}")
    
    total_steps = n_steps + burn_in_steps
    
    # Getting the truncated normal samples
    lower, upper = 0.01, 25.0
    
    a = (lower - mu0) / sigma0_init
    b = (upper - mu0) / sigma0_init
    
    X0_samples = truncnorm.rvs(
        a, b,
        loc=mu0,
        scale=sigma0_init,
        size=n_scenarios
    )
    
    # Creating storage for the variables
    X_paths_full = np.zeros((n_scenarios, total_steps))
    X_paths_full[:, 0] = X0_samples
    X_current = X0_samples.copy()
    
    # Drawing all the nomral variables at once to speed up performance
    Z = np.random.normal(size=(n_scenarios, total_steps - 1))
    
    # Looping over the time steps
    for t in range(1, total_steps):
        # Getting z variable
        z = Z[:, t - 1]
        
        # Drift and diffusion variables
        drift = kappa * (theta - X_current) * dt
        diffusion = sigma * np.sqrt(np.maximum(X_current, 0.0)) * np.sqrt(dt) * z
        
        # Calculating next rate using current rate, drift, and diffusion
        X_next = X_current + drift + diffusion
        
        # Enforce positivity
        X_next = np.maximum(X_next, 0.0)
        
        # Updating current
        X_paths_full[:, t] = X_next
        X_current = X_next
    
    # Converting to decimal format
    X_paths_full = X_paths_full / 100.0
    
    return X_paths_full

### Running the Simulations

In [128]:
# =============================================================================
# Sorting out saving directory
# =============================================================================

output_dir = 'black_scholes_simulation_data'
os.makedirs(output_dir, exist_ok=True)

total_start = time.time()

print(f"\nGenerating {n_simulations:,} simulations × {n_days} days")

# =============================================================================
# Running GARCH
# =============================================================================

print(f"Running GARCH simulation...")
sim_start = time.time()

# CRITICAL: Use deterministic seed
simulation_seed = MASTER_SEED + 5000

S_paths, sigma_paths = simulate_garch(
    S0=S0,
    n_days=n_days,
    n_scenarios=n_simulations,
    mu=mu_adjusted,
    omega=omega,
    alpha=alpha,
    beta=beta,
    nu=nu,
    seed=simulation_seed,
    burn_in=burn_in_period
)

print(f"GARCH simulation completed in {time.time() - sim_start:.1f}s")

# =============================================================================
# Running CIR Model
# =============================================================================

print(f"Running CIR simulation...")
sim_start = time.time()

# Deterministic seed for CIR
cir_seed = MASTER_SEED + 7000

interest_paths = simulate_CIR_paths(
    n_scenarios=n_simulations,
    n_steps=n_days,
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    mu0=ir_mean,
    sigma0_init=ir_std,
    dt=1/252,
    seed=cir_seed,
    burn_in_days=burn_in_period
)

print(f"CIR simulation completed in {time.time() - sim_start:.1f}s")

# =============================================================================
# Building a vectorized Dataset 
# =============================================================================

print(f"\nBuilding dataset...")
build_start = time.time()


n_rows = n_simulations * n_days

# Create arrays for each column
simulation_col = np.repeat(np.arange(n_simulations), n_days + burn_in_period)
day_col = np.tile(np.arange(-burn_in_period, n_days), n_simulations)
S_col = S_paths.flatten()
K_col = np.repeat(K_assignments, n_days + burn_in_period)
interest_col = interest_paths.flatten()

# Create T column:
# End at exactly 1 year remaining
T_sequence = 1 + (n_days + burn_in_period - 1 - np.arange(n_days + burn_in_period)) / days_per_year
T_col = np.tile(T_sequence, n_simulations)

# Convert volatility to annualized percentage
sigma_col = sigma_paths.flatten() * np.sqrt(days_per_year) / 100

print(f"Arrays built in {time.time() - build_start:.1f}s")

# =============================================================================
# Creating the DataFrame
# =============================================================================

print(f"Creating DataFrame...")
df_start = time.time()

# Debugging
print(simulation_col.shape)
print(day_col.shape)
print(S_col.shape)
print(K_col.shape)
print(T_col.shape)
print(sigma_col.shape)
print(interest_col.shape)


df = pd.DataFrame({
    'simulation': simulation_col,
    'day': day_col,
    'S': S_col,
    'K': K_col,
    'T': T_col,
    'sigma': sigma_col,
    'r': interest_col
})

# Sort by simulation, then descending T (day ascending)
df = df.sort_values(
    by=['simulation', 'day'],
    ascending=[True, True]
).reset_index(drop=True)

print(f"DataFrame created in {time.time() - df_start:.1f}s")

total_time = time.time() - total_start
print(f"\n" + "=" * 70)
print(f"SIMULATION COMPLETE! Total time: {total_time / 60:.1f} minutes")
print("=" * 70)

# =============================================================================
# Saving
# =============================================================================

print("\n" + "=" * 70)
print("STEP 5: FINAL DATASET")
print("=" * 70)

print("\nStatistics:")
print(df.describe())

# Compute final hash
print("\nComputing dataset hash...")
final_hash = compute_dataframe_hash(df)
print(f"Final dataset hash: {final_hash}")

# Save final file as CSV
print("\nSaving final dataset...")
save_start = time.time()
df.to_csv('black_scholes_simulation_data_T5.csv', index=False)
save_time = time.time() - save_start

file_size_gb = os.path.getsize('black_scholes_simulation_data_T5.csv') / 1e9
print(f"Saved to black_scholes_simulation_data_T5.csv in {save_time:.1f}s")
print(f"File size: {file_size_gb:.2f} GB")

# =============================================================================
# STEP 7: SAVE METADATA AND VERIFICATION INFO
# =============================================================================

print("\n" + "=" * 70)
print("STEP 6: SAVING METADATA FOR REPRODUCIBILITY")
print("=" * 70)

# Add hashes and final statistics to metadata
metadata['final_hash'] = final_hash
metadata['total_rows'] = int(len(df))
metadata['file_size_gb'] = float(file_size_gb)
metadata['total_generation_time_minutes'] = float(total_time / 60)
metadata['simulation_seed'] = int(simulation_seed)
metadata['cir_seed'] = int(cir_seed)

# Save metadata
save_metadata(metadata, f'{output_dir}/dataset_metadata.json')

# Also save a verification file with just the hash
verification = {
    'master_seed': MASTER_SEED,
    'simulation_seed': simulation_seed,
    'final_hash': final_hash,
    'total_rows': int(len(df)),
    'n_simulations': int(n_simulations),
    'T_maturity': int(T_maturity),
    'n_days': int(n_days),
    'cir_seed' : int(cir_seed),
    'instructions': f'Run the generation script with MASTER_SEED={MASTER_SEED} to reproduce this exact dataset'
}

with open('REPRODUCIBILITY_INFO.json', 'w') as f:
    json.dump(verification, f, indent=2)

print(f"\nVerification info saved to REPRODUCIBILITY_INFO.json")

print("\n" + "=" * 70)
print("DONE!")
print("=" * 70)
print("\nTo reproduce this dataset on another computer:")
print(f"1. Use MASTER_SEED = {MASTER_SEED}")
print(f"2. Expected final hash: {final_hash}")
print(f"3. Expected total rows: {len(df):,}")
print(f"4. Expected file size: {file_size_gb:.2f} GB")
print("=" * 70)

# Clean up memory
del S_paths, sigma_paths, interest_paths


Generating 10,000 simulations × 1009 days
Running GARCH simulation...
GARCH simulation completed in 2.2s
Running CIR simulation...
Total burn in steps: 90
CIR simulation completed in 0.7s

Building dataset...
Arrays built in 0.3s
Creating DataFrame...
(10990000,)
(10990000,)
(10990000,)
(10990000,)
(10990000,)
(10990000,)
(10990000,)
DataFrame created in 1.7s

SIMULATION COMPLETE! Total time: 0.1 minutes

STEP 5: FINAL DATASET

Statistics:
         simulation           day             S             K             T  \
count  1.099000e+07  1.099000e+07  1.099000e+07  1.099000e+07  1.099000e+07   
mean   4.999500e+03  4.590000e+02  1.054757e+04  8.579362e+03  3.178571e+00   
std    2.886751e+03  3.172539e+02  3.082411e+03  2.365119e+03  1.258944e+00   
min    0.000000e+00 -9.000000e+01  2.503122e+03  4.903800e+03  1.000000e+00   
25%    2.499750e+03  1.840000e+02  8.448440e+03  6.538400e+03  2.087302e+00   
50%    4.999500e+03  4.590000e+02  9.764539e+03  8.990300e+03  3.178571e+00   
75

### Adding the Black-Scholes-Merton Price 
Price is given by:
$$C(S,t)=N(d_1)S-N(d_2)Ke^{-rT}$$
Where
1. $C(S,t)$ is the call price
2. $N()$ is the Normal CDF
3. $T$ is time to maturity
4. $S$ is stock price
5. $K$ is strike price
6. $r$ is the risk-free rate
7. $\sigma$ is volatility
and 
$$d_1 = \frac{\text{ln}\left(\frac{S}{K}\right)+\left(r+\frac{\sigma^2}{2}\right)T}{\sigma \sqrt{T}}$$
and
$$d_2 = d_1 - \sigma\sqrt{T}$$

In [129]:
def add_black_scholes_price(df):
    """
    Adds a Black-Scholes call price column ('bs_price') to the dataframe.
    
    Required columns:
    S, K, T, sigma, r
    
    r is assumed to be in percent and will be divided by 100.
    """
    
    r = df['r'] 
    S = df['S']
    K = df['K']
    T = df['T']
    sigma = df['sigma']
    
    # Avoid division by zero warnings
    epsilon = 1e-12
    sigma = np.maximum(sigma, epsilon)
    T = np.maximum(T, epsilon)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    df['bs_price'] = (
        S * norm.cdf(d1) - 
        K * np.exp(-r * T) * norm.cdf(d2)
    )
    
    return df

input_file = "black_scholes_simulation_data_T5.csv"
output_file = "black_scholes_simulation_data_T5_with_price.csv"

chunksize = 1_000_000  # adjust based on memory

with pd.read_csv(input_file, chunksize=chunksize) as reader:
    for i, chunk in enumerate(reader):
        chunk = add_black_scholes_price(chunk)

        # Write header only for first chunk
        chunk.to_csv(
            output_file,
            mode='w' if i == 0 else 'a',
            header=(i == 0),
            index=False
        )

In [152]:
totaldf = pd.read_csv("black_scholes_simulation_data_T5_with_price.csv")
totaldf.head(1100)

,simulation,day,S,K,T,sigma,r,bs_price
0,0,-90,8173.000000,9807.6,5.357143,0.159915,0.031455,1152.325910
1,0,-89,8194.737845,9807.6,5.353175,0.149432,0.032077,1096.775253
2,0,-88,8127.574672,9807.6,5.349206,0.148336,0.032875,1065.130990
3,0,-87,8181.645358,9807.6,5.345238,0.143305,0.032790,1055.308480
4,0,-86,8126.365310,9807.6,5.341270,0.140573,0.032455,997.326131
...,...,...,...,...,...,...,...,...
1095,0,1005,17837.713178,9807.6,1.011905,0.193794,0.019392,8221.215945
1096,0,1006,18069.359785,9807.6,1.007937,0.193002,0.019355,8451.622268
1097,0,1007,17876.002345,9807.6,1.003968,0.189548,0.018716,8251.336706
1098,0,1008,18032.215183,9807.6,1.000000,0.181586,0.018705,8406.529869


# WORK IN PROGRESS CODE BELOW

In [153]:
def rolling_mean(x, window, lag=0):
    """
    Parameters
    ----------
    x : ndarray (1D)
    window : int
        Size of moving window
    lag : int
        Number of periods to lag the result
        lag=0 -> standard rolling mean
        lag=1 -> excludes current value
        lag=k -> shifted back by k periods
    
    Returns
    -------
    ndarray of same length as x
    """
    
    x = np.asarray(x, dtype=float)
    n = len(x)
    
    result = np.full(n, np.nan)
    
    if window <= 0:
        raise ValueError("window must be positive")
    if lag < 0:
        raise ValueError("lag must be >= 0")
    if window + lag > n:
        return result  # cannot compute anything
    
    # cumulative sum trick
    cumsum = np.cumsum(np.insert(x, 0, 0))
    
    # rolling sums (no lag yet)
    rolling_sum = cumsum[window:] - cumsum[:-window]
    rolling_mean = rolling_sum / window
    
    # place into correct positions with lag
    start = window - 1 + lag
    end = start + len(rolling_mean)
    
    result[start:end] = rolling_mean[:n-start]
    
    return result

def get_moving_averages(df:pd.DataFrame):
    # Making sure that the dataframe has the required column names to get deltas 
    try:
        assert("S" in df.columns)
        assert("sigma" in df.columns)
        assert("bs_price" in df.columns)
        assert("r" in df.columns)
    except:
        print("Not all required columns are in the dataframe")
        return 
    
    # Getting the number of simulations and the days per simulation with the burn-in period included 
    num_simulations = df["simulation"].max()
    days_per_simulation = n_days + burn_in_period 
    
    #######################################
    # Getting the lag variables
    #######################################
    
    # No lag, 5 day window
    ma_S_window_5_lag_0 = np.zeros(df.shape[0])
    ma_sigma_window_5_lag_0 = np.zeros(df.shape[0])
    ma_BSprice_window_5_lag_0 = np.zeros(df.shape[0])
    ma_r_window_5_lag_0 = np.zeros(df.shape[0])
    
    # 5 day lag, 15 day window
    ma_S_window_15_lag_5 = np.zeros(df.shape[0])
    ma_sigma_window_15_lag_5 = np.zeros(df.shape[0])
    ma_BSprice_window_15_lag_5 = np.zeros(df.shape[0])
    ma_r_window_15_lag_5 = np.zeros(df.shape[0])
    
    # 10 day lag, 30 day window
    ma_S_window_30_lag_10 = np.zeros(df.shape[0])
    ma_sigma_window_30_lag_10 = np.zeros(df.shape[0])
    ma_BSprice_window_30_lag_10 = np.zeros(df.shape[0])
    ma_r_window_30_lag_10 = np.zeros(df.shape[0])

    # Looping over every simulation
    for i in range(num_simulations + 1):
        sim_chunk_start = days_per_simulation * i
        sim_chunk_end = days_per_simulation * (i+1)
        
        # Getting a chunk of simulation data
        simulation = df[sim_chunk_start : sim_chunk_end]
        
        # Getting the 5-day window ones
        ma_S_window_5_lag_0[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['S'], 5)
        ma_sigma_window_5_lag_0[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['sigma'], 5)
        ma_BSprice_window_5_lag_0[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['bs_price'], 5)
        ma_r_window_5_lag_0[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['r'], 5) 
        
        # Getting the 30 day window ones
        ma_S_window_15_lag_5[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['S'], 15, 5)
        ma_sigma_window_15_lag_5[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['sigma'], 15, 5)
        ma_BSprice_window_15_lag_5[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['bs_price'], 15, 5)
        ma_r_window_15_lag_5[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['r'], 15, 5) 
        
        # Getting the 90 day window ones 
        ma_S_window_30_lag_10[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['S'], 30, 10)
        ma_sigma_window_30_lag_10[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['sigma'], 30, 10)
        ma_BSprice_window_30_lag_10[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['bs_price'], 30, 10)
        ma_r_window_30_lag_10[sim_chunk_start : sim_chunk_end] = rolling_mean(simulation['r'], 30, 10) 
    
    # Setting columns in dataframe
    df["ma_S_window_5_lag_0"],  df["ma_sigma_window_5_lag_0"], df["ma_BSprice_window_5_lag_0"], df["ma_r_window_5_lag_0"], \
        df["ma_S_window_15_lag_5"],df["ma_sigma_window_15_lag_5"],df["ma_BSprice_window_15_lag_5"],df["ma_r_window_15_lag_5"], \
            df["ma_S_window_30_lag_10"],df["ma_sigma_window_30_lag_10"],df["ma_BSprice_window_30_lag_10"],df["ma_r_window_30_lag_10"] = \
                ma_S_window_5_lag_0, ma_sigma_window_5_lag_0, ma_BSprice_window_5_lag_0, ma_r_window_5_lag_0, \
                    ma_S_window_15_lag_5, ma_sigma_window_15_lag_5, ma_BSprice_window_15_lag_5, ma_r_window_15_lag_5, \
                        ma_S_window_30_lag_10, ma_sigma_window_30_lag_10, ma_BSprice_window_30_lag_10, ma_r_window_30_lag_10




def get_detlas(df:pd.DataFrame):
    # Making sure that the dataframe has the required column names to get deltas 
    try:
        assert("S" in df.columns)
        assert("sigma" in df.columns)
        assert("bs_price" in df.columns)
        assert("r" in df.columns)
    except:
        print("Not all required columns are in the dataframe")
        return 
    
    # Getting the number of simulations and the days per simulation with the burn-in period included 
    num_simulations = df["simulation"].max()
    days_per_simulation = n_days + burn_in_period 
    
    # Getting variables to save data
    dS = np.zeros(df.shape[0])
    dsigma = np.zeros(df.shape[0])
    dBS_price = np.zeros(df.shape[0])
    dr = np.zeros(df.shape[0])
    
    # Looping over every simulation
    for i in range(num_simulations + 1):
        sim_chunk_start = days_per_simulation * i
        sim_chunk_end = days_per_simulation * (i+1)
        
        # Getting a chunk of simulation data
        simulation = df[sim_chunk_start : sim_chunk_end]
        
        # Transforming the simulation data and adding it to the variables 
        dS[sim_chunk_start : sim_chunk_end] = simulation["S"] - simulation["S"].shift(1)
        dsigma[sim_chunk_start : sim_chunk_end] = simulation["sigma"] - simulation["sigma"].shift(1)
        dBS_price[sim_chunk_start : sim_chunk_end] = simulation["bs_price"] - simulation["bs_price"].shift(1)
        dr[sim_chunk_start : sim_chunk_end] = simulation["r"] - simulation["r"].shift(1)
    
    # Adding the numpy arrays to the dataframe
    df["dS"], df["dsigma"], df["dBS_price"], df["dr"] = dS, dsigma, dBS_price, dr
    
    return df
    
get_detlas(totaldf)
get_moving_averages(totaldf)

In [154]:
def add_greeks(df, option_type="call"):
    """
    Adds Greeks_Delta, Greeks_Gamma, Greeks_Vega,
    Greeks_Theta, Greeks_Rho columns to dataframe.
    
    Required columns:
        S, K, T, sigma, r
    """
    
    S = df["S"]
    K = df["K"]
    T = np.maximum(df["T"], 1e-12)   # avoid division by zero
    sigma = df["sigma"]
    r = df["r"]
    
    sqrtT = np.sqrt(T)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    
    # Delta
    df["Greeks_Delta"] = norm.cdf(d1)

    # Gamma
    df["Greeks_Gamma"] = norm.pdf(d1) / (S * sigma * sqrtT)

    # Vega (per 1.0 volatility change)
    df["Greeks_Vega"] = S * norm.pdf(d1) * sqrtT

    # Theta (per day)
    df["Greeks_Theta_daily"] = (
        - (S * norm.pdf(d1) * sigma) / (2 * sqrtT)
        - r * K * np.exp(-r * T) * norm.cdf(d2)
    ) / 365
    
    # Rho
    df["Greeks_Rho"] = K * T * np.exp(-r * T) * norm.cdf(d2)

add_greeks(totaldf)

In [ ]:
from joblib import Parallel, delayed
from numpy.lib.stride_tricks import sliding_window_view
from numba import njit

# ── Constants ─────────────────────────────────────────────────────────────────
N_BINS = 7
WINDOW = 89
SCALES = np.logspace(np.log10(2), np.log10(45), N_BINS)
WAVELET_NAME  = 'cmor1.5-1.0'

def compute_wavelet_features_vectorized(df):

    df = df.sort_values(["simulation", "day"]).copy()
    feature_cols = [f"wavelet_bin_{i}" for i in range(N_BINS * N_BINS)]
    df[feature_cols] = 0.0  # ensures no NaNs

    for sim in df["simulation"].unique():

        sim_mask = df["simulation"] == sim
        sim_df = df.loc[sim_mask]

        prices = sim_df["dBS_price"].values
        days = sim_df["day"].values
        n = len(prices)

        if n < WINDOW:
            continue

        # --------------------------------------------------
        # 1️⃣ Compute CWT ONCE for entire simulation
        # --------------------------------------------------
        coeffs, freqs = pywt.cwt(prices, SCALES, WAVELET_NAME)

        # Energy (magnitude squared of complex coeffs)
        energy = np.abs(coeffs) ** 2  # shape: (n_scales, n_days)

        # --------------------------------------------------
        # 2️⃣ Rolling window extraction (vectorized)
        # --------------------------------------------------
        # shape becomes (n_scales, n_windows, WINDOW)
        energy_windows = sliding_window_view(energy, WINDOW, axis=1)

        # Eligible days (no NaNs for day >= WINDOW)
        eligible_idx = np.where(days >= WINDOW)[0]
        if len(eligible_idx) == 0:
            continue

        window_positions = eligible_idx - (WINDOW - 1)
        energy_windows = energy_windows[:, window_positions, :]

        # --------------------------------------------------
        # 3️⃣ Convert frequencies → periods
        # --------------------------------------------------
        periods = 1.0 / freqs

        # --------------------------------------------------
        # 4️⃣ Create log bin edges
        # --------------------------------------------------
        p_edges = np.logspace(
            np.log10(periods.min()),
            np.log10(periods.max()),
            N_BINS + 1
        )

        times = np.arange(1, WINDOW + 1)
        t_edges = np.logspace(
            np.log10(times.min()),
            np.log10(times.max()),
            N_BINS + 1
        )

        # Bin indices
        p_bin_idx = np.digitize(periods, p_edges) - 1
        p_bin_idx = np.clip(p_bin_idx, 0, N_BINS - 1)

        t_bin_idx = np.digitize(times, t_edges) - 1
        t_bin_idx = np.clip(t_bin_idx, 0, N_BINS - 1)

        # --------------------------------------------------
        # 5️⃣ Vectorized 2D bin averaging
        # --------------------------------------------------
        n_windows = energy_windows.shape[1]
        features = np.zeros((n_windows, N_BINS * N_BINS))

        for i in range(N_BINS):
            scale_mask = (p_bin_idx == i)

            if not np.any(scale_mask):
                continue

            scale_slice = energy_windows[scale_mask]  # select scales

            for j in range(N_BINS):
                time_mask = (t_bin_idx == j)

                if not np.any(time_mask):
                    continue

                # Mean over selected scales and time indices
                bin_energy = scale_slice[:, :, time_mask].mean(axis=(0, 2))
                features[:, i * N_BINS + j] = bin_energy

        # --------------------------------------------------
        # 6️⃣ Write back
        # --------------------------------------------------
        df.loc[
            sim_mask & (df["day"].isin(days[eligible_idx])),
            feature_cols
        ] = features

    return df

compute_wavelet_features_vectorized(totaldf)

In [147]:
totaldf = totaldf[totaldf['day'] >= 0]
totaldf = totaldf.reset_index()

In [148]:
totaldf.head(92)

,index,simulation,day,S,K,T,sigma,r,bs_price,dS,...,wavelet_bin_p5_t4,wavelet_bin_p5_t5,wavelet_bin_p5_t6,wavelet_bin_p6_t0,wavelet_bin_p6_t1,wavelet_bin_p6_t2,wavelet_bin_p6_t3,wavelet_bin_p6_t4,wavelet_bin_p6_t5,wavelet_bin_p6_t6
0,90,0,0,7788.934601,9807.6,5.000000,0.101772,0.027115,416.259465,40.682990,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,91,0,1,7758.566777,9807.6,4.996032,0.101484,0.026563,395.518217,-30.367824,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,92,0,2,7739.706462,9807.6,4.992063,0.099485,0.026244,371.289060,-18.860315,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,93,0,3,7731.784705,9807.6,4.988095,0.096776,0.025686,344.081885,-7.921757,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,94,0,4,7795.315086,9807.6,4.984127,0.104399,0.025592,414.702014,63.530381,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,177,0,87,7985.913876,9807.6,4.654762,0.163189,0.025566,840.285694,170.287790,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,178,0,88,7962.840795,9807.6,4.650794,0.152943,0.024773,748.164233,-23.073081,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,179,0,89,7930.619120,9807.6,4.646825,0.145041,0.024562,676.823344,-32.221675,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,180,0,90,7943.855324,9807.6,4.642857,0.135982,0.024879,625.037104,13.236204,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
totaldf.head(92)

,simulation,day,S,K,T,sigma,r,bs_price,dS,dsigma,...,wavelet_bin_p5_t4,wavelet_bin_p5_t5,wavelet_bin_p5_t6,wavelet_bin_p6_t0,wavelet_bin_p6_t1,wavelet_bin_p6_t2,wavelet_bin_p6_t3,wavelet_bin_p6_t4,wavelet_bin_p6_t5,wavelet_bin_p6_t6
0,0,-90,8173.000000,9807.6,5.357143,0.159915,0.031455,1152.325910,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,-89,8194.737845,9807.6,5.353175,0.149432,0.031184,1080.161280,21.737845,-0.010482,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,-88,8127.574672,9807.6,5.349206,0.148336,0.031153,1033.704999,-67.163173,-0.001096,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,-87,8181.645358,9807.6,5.345238,0.143305,0.031309,1027.687026,54.070686,-0.005031,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,-86,8126.365310,9807.6,5.341270,0.140573,0.031708,983.634262,-55.280048,-0.002733,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,0,-3,7718.199802,9807.6,5.011905,0.108024,0.032823,507.619216,-20.657234,-0.003335,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,0,-2,7743.651759,9807.6,5.007937,0.105036,0.032804,497.262837,25.451957,-0.002987,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,0,-1,7748.251611,9807.6,5.003968,0.101144,0.032750,471.420919,4.599852,-0.003892,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,0,0,7788.934601,9807.6,5.000000,0.101772,0.032193,484.412872,40.682990,0.000628,...,308.799642,339.541865,703.902618,566.646697,456.527867,486.847657,523.676678,551.774505,552.153649,734.422643


In [ ]:
import numpy as np
import plotly.graph_objects as go
import pywt

def plot_wavelet_3d_plotly(df, simulation_num, day_num, wavelet='cmor1.5-1.0', window=90, scale=45):
    """
    Plot an interactive 3D surface of the CWT energy for a given simulation and day.
    """
    # Extract the 90-day window
    sim_data = df[df['simulation'] == simulation_num].sort_values('day')
    window_data = sim_data[sim_data['day'] < day_num].tail(window)['dBS_price'].values

    if len(window_data) < window:
        raise ValueError(f"Not enough data: only {len(window_data)} days available before day {day_num}")

    # CWT
    scales = np.arange(1, scale + 1)
    center_freq = pywt.central_frequency(wavelet)
    periods = scales / center_freq  # Convert scales to periods in days

    coeffs, _ = pywt.cwt(window_data, scales, wavelet)
    energy = np.abs(coeffs) ** 2  # shape: (n_scales, 90)

    # Axes
    time_axis = np.arange(1, window + 1)  # 1 to 90

    # Plot
    fig = go.Figure(data=[go.Surface(
        x=time_axis,
        y=periods,
        z=energy,
        colorscale='Plasma',
        colorbar=dict(title='Energy')
    )])

    fig.update_layout(
        title=f'Wavelet Transform Energy — Simulation {simulation_num}, Day {day_num}',
        scene=dict(
            xaxis_title='Time (days in window)',
            yaxis_title='Period (days)',
            zaxis_title='Energy',
        ),
        width=900,
        height=700
    )

    fig.show()

In [ ]:
def plot_wavelet_bins_3d(df, simulation_num, day_num, n_bins=7, scale=45, window=90,
                          x_axis='log', y_axis='log', wavelet='cmor1.5-1.0'):
    """
    Plot the 49 binned wavelet energies as a 3D bar chart for a given simulation and day.
    
    Parameters:
        df             : DataFrame with wavelet bin columns (wavelet_bin_p{p}_t{t})
        simulation_num : Simulation number to plot
        day_num        : Day number to plot
        n_bins         : Number of bins per axis (default 7)
        scale          : Max scale used in CWT (default 45)
        window         : Window size in days (default 90)
        x_axis         : 'log' or 'linear' for time axis
        y_axis         : 'log' or 'linear' for period axis
        wavelet        : Wavelet used (needed to reconstruct period bins)
    """
    import pywt

    # --- Reconstruct bin edges ---
    scales = np.arange(1, scale + 1)
    center_freq = pywt.central_frequency(wavelet)
    periods = scales / center_freq

    period_bins = np.logspace(np.log10(periods.min()), np.log10(periods.max()), n_bins + 1)
    time_bins   = np.logspace(np.log10(1), np.log10(window), n_bins + 1)

    # Bin centers
    period_centers = (period_bins[:-1] + period_bins[1:]) / 2
    time_centers   = (time_bins[:-1]   + time_bins[1:])   / 2

    # Bin widths (for bar sizing)
    period_widths = period_bins[1:] - period_bins[:-1]
    time_widths   = time_bins[1:]   - time_bins[:-1]

    # --- Extract row ---
    row = df[(df['simulation'] == simulation_num) & (df['day'] == day_num)]
    if row.empty:
        raise ValueError(f"No data found for simulation {simulation_num}, day {day_num}")
    row = row.iloc[0]

    # --- Build bar data ---
    x, y, z_base, z_top, dx, dy, colors = [], [], [], [], [], [], []

    energy_values = np.zeros((n_bins, n_bins))
    for p in range(n_bins):
        for t in range(n_bins):
            col = f'wavelet_bin_p{p}_t{t}'
            energy_values[p, t] = row[col] if col in row.index and not np.isnan(row[col]) else 0.0

    e_min, e_max = energy_values.min(), energy_values.max()

    # Colorscale mapping
    def energy_to_color(e):
        norm = (e - e_min) / (e_max - e_min + 1e-12)
        # Plasma colorscale: map norm to RGB roughly
        return norm

    for p in range(n_bins):
        for t in range(n_bins):
            x.append(time_centers[t])
            y.append(period_centers[p])
            z_base.append(0)
            z_top.append(energy_values[p, t])
            dx.append(time_widths[t] * 0.8)
            dy.append(period_widths[p] * 0.8)

    x, y, z_top = np.array(x), np.array(y), np.array(z_top)
    dx, dy = np.array(dx), np.array(dy)

    # Normalize for color
    norm_energy = (z_top - e_min) / (e_max - e_min + 1e-12)

    # --- Build mesh bars manually via go.Mesh3d for each bar ---
    # More efficient: use a single scatter3d with error bars trick,
    # but for true 3D boxes we build one Mesh3d per bar
    def make_bar_mesh(cx, cy, cz_top, ddx, ddy, color_val):
        """Return vertices and faces for a single 3D box."""
        x0, x1 = cx - ddx/2, cx + ddx/2
        y0, y1 = cy - ddy/2, cy + ddy/2
        z0, z1 = 0, cz_top

        vx = [x0,x1,x1,x0, x0,x1,x1,x0]
        vy = [y0,y0,y1,y1, y0,y0,y1,y1]
        vz = [z0,z0,z0,z0, z1,z1,z1,z1]

        i = [0,0,0,4,4,4, 1,2,3,5,6,7]
        j = [1,3,4,5,7,0, 5,6,7,1,2,3]
        k = [2,7,5,6,3,3, 2,7,4,6,3,0]

        return vx, vy, vz, i, j, k

    import plotly.colors as pc
    colorscale = pc.get_colorscale('Plasma')

    def sample_colorscale(val):
        return pc.sample_colorscale('Plasma', val)[0]

    traces = []
    for idx in range(len(x)):
        vx, vy, vz, fi, fj, fk = make_bar_mesh(x[idx], y[idx], z_top[idx], dx[idx], dy[idx], norm_energy[idx])
        color = sample_colorscale(float(norm_energy[idx]))
        traces.append(go.Mesh3d(
            x=vx, y=vy, z=vz,
            i=fi, j=fj, k=fk,
            color=color,
            opacity=0.85,
            showscale=False,
            hovertemplate=(
                f"Time center: {x[idx]:.1f} days<br>"
                f"Period center: {y[idx]:.2f} days<br>"
                f"Energy: {z_top[idx]:.4f}<extra></extra>"
            )
        ))

    # Dummy scatter for colorbar
    traces.append(go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode='markers',
        marker=dict(
            colorscale='Plasma',
            color=[e_min, e_max],
            colorbar=dict(title='Energy'),
            size=0
        ),
        showlegend=False
    ))

    # --- Axis type ---
    xaxis_cfg = dict(title='Time (days in window)', type='log' if x_axis == 'log' else 'linear')
    yaxis_cfg = dict(title='Period (days)',          type='log' if y_axis == 'log' else 'linear')
    zaxis_cfg = dict(title='Mean Energy')

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=f'Binned Wavelet Energy — Simulation {simulation_num}, Day {day_num}',
        scene=dict(
            xaxis=xaxis_cfg,
            yaxis=yaxis_cfg,
            zaxis=zaxis_cfg,
        ),
        width=950,
        height=750
    )

    fig.show()